In [0]:
"""
Basic Percentage Calculation

Defines a simple add_percent() function that converts a fraction to a percentage.
Includes 2 tests: basic calculation and zero edge case.
"""

def add_percent(part, total):
    return round(part / total * 100, 1)

def test_add_percent_basic():
    # Test: 1/4 should equal 25%
    assert add_percent(1, 4) == 25.0

def test_add_percent_zero():
    # Test: 0/4 should equal 0% (edge case: zero numerator)
    assert add_percent(0, 4) == 0.0

In [0]:
"""
Null Percentage Calculator for DataFrames

Defines null_percentage() function to calculate the percentage of null values
in a specified column. Includes tests for:
- DataFrames with no nulls (should return 0%)
- DataFrames with some nulls (should return correct percentage)
"""

from pyspark.sql.functions import col

def null_percentage(df, column_name):
    total = df.count()
    nulls = df.filter(col(column_name).isNull()).count()
    return round(nulls / total * 100, 1)

def test_null_percentage_no_nulls():
    # Test: Should return 0% when no nulls present
    data = [(1, "Alice"), (2, "Ben")]
    df = spark.createDataFrame(data, ["id", "name"])
    assert null_percentage(df, "name") == 0.0

def test_null_percentage_some_nulls():
    # Test: Should return 50% when half the values are null
    data = [(1, "Alice"), (2, None)]
    df = spark.createDataFrame(data, ["id", "name"])
    assert null_percentage(df, "name") == 50.0

In [0]:
"""
Edge Case Test: Empty DataFrame

Tests that null_percentage() properly handles an empty DataFrame.
Expects a ZeroDivisionError to be raised (division by zero when count is 0).
"""

def test_null_percentage_empty_dataframe():
    # Test: Should raise ZeroDivisionError for empty DataFrame (edge case)
    data = []
    schema = "id INT, name STRING"
    df = spark.createDataFrame(data, schema)
    try:
        null_percentage(df, "name")
        assert False, "Expected a ZeroDivisionError but none was raised"
    except ZeroDivisionError:
        pass

In [0]:
# ====================================================================
# FINAL TEST RUNNER - Run All Tests with Pytest
# ====================================================================
"""
Pytest Test Discovery and Execution

Uses pytest to automatically discover all functions starting with 'test_'
and runs them with verbose output. This is the professional approach
for running tests with detailed reporting.
# ====================================================================
# FINAL TEST RUNNER - Run All Tests with Pytest
# ====================================================================
"""

import pytest
import inspect

print("\n" + "="*70)
print("Running All Unit Tests with Pytest")
print("="*70 + "\n")

# Discover all test functions in the current namespace
test_functions = {
    name: obj for name, obj in globals().items()
    if name.startswith('test_') and callable(obj)
}

print(f"Found {len(test_functions)} test functions:\n")

passed = 0
failed = 0
errors = []

for test_name in sorted(test_functions.keys()):
    test_func = test_functions[test_name]
    try:
        # Check if test requires sample_df fixture
        sig = inspect.signature(test_func)
        if 'sample_df' in sig.parameters:
            test_func(sample_df())
        else:
            test_func()
        print(f"✓ {test_name} PASSED")
        passed += 1
    except AssertionError as e:
        print(f"✗ {test_name} FAILED: {str(e)}")
        failed += 1
        errors.append((test_name, str(e)))
    except Exception as e:
        print(f"✗ {test_name} ERROR: {str(e)}")
        failed += 1
        errors.append((test_name, str(e)))

print("\n" + "="*70)
print(f"Results: {passed} passed, {failed} failed")
if failed == 0:
    print("✓ All Tests Passed Successfully!")
else:
    print(f"✗ {failed} Test(s) Failed")
    print("\nFailure Details:")
    for test_name, error in errors:
        print(f"  - {test_name}: {error}")
print("="*70)

In [0]:
"""
Fixture-Based Testing with Sample Data

Defines a sample_df() function that creates test data with 3 employees.
Includes tests that use this sample data:
- test_salary_null_check: Verifies null detection (1 null out of 3)
- test_row_count: Verifies DataFrame row count
"""

import pytest

def sample_df():
    data = [(1, "Alice", 55000), (2, "Ben", None), (3, "Cara", 62000)]
    return spark.createDataFrame(data, ["id", "name", "salary"])

def test_salary_null_check(sample_df):
    # Test: Should detect 33.3% nulls in salary column (1 out of 3)
    assert null_percentage(sample_df, "salary") == pytest.approx(33.3, 0.1)

def test_row_count(sample_df):
    # Test: Sample DataFrame should have exactly 3 rows
    assert sample_df.count() == 3

# Tests will be run by pytest at the end

In [0]:
"""
Range Validation for Data Quality

Defines range_check_failures() to count values outside acceptable range.
Includes 2 tests:
- test_range_check_finds_failures: Verifies detection of out-of-range values
- test_range_check_no_false_positives: Verifies no false alarms on valid data
"""

def range_check_failures(df, column_name, low, high):
    return df.filter((col(column_name) < low) | (col(column_name) > high)).count()

def test_range_check_finds_failures():
    # Test: Should detect 2 out-of-range salaries (negative and too high)
    data = [(1, 55000), (2, -5000), (3, 62000), (4, 5000000)]
    df = spark.createDataFrame(data, ["id", "salary"])
    assert range_check_failures(df, "salary", 0, 1000000) == 2

def test_range_check_no_false_positives():
    # Test: Should find 0 failures when all salaries are in valid range
    data = [(1, 50000), (2, 60000)]
    df = spark.createDataFrame(data, ["id", "salary"])
    assert range_check_failures(df, "salary", 0, 1000000) == 0

# Tests will be run by pytest at the end

